In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_10_dynamical_correlations_and_transport.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 10 — Dynamical Correlations, Linear Response, and Transport Coefficients

_Session 88082df3 · Estimated time: ~115 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Derive static linear response from perturbation theory and explain why the static structure factor S(k) governs the linear density response to an external field.
- Apply stationarity, Cauchy-Schwarz bounds, the Onsager regression hypothesis, and the Fluctuation-Dissipation Theorem to equilibrium time correlation functions.
- Relate the intermediate scattering function F(k, t) and dynamical structure factor S(k, ω) to inelastic scattering, collective sound modes, Kramers-Kronig relations, and quantum detailed balance.
- Compute self-diffusion coefficients via both the Einstein unwrapped mean-squared displacement (MSD) route and the Green-Kubo velocity autocorrelation (VACF) integral, and explain Alder-Wainwright hydrodynamic long-time tails.


In [2]:
ws.render_section('sec_static_perturbation_and_time_correlations')

## 1. Static Perturbation Theory, Linear Response, and Time Correlation Functions

Why perform **Molecular Dynamics (MD)** rather than **Monte Carlo (MC)**? While both methods sample static equilibrium averages, **only MD propagates Newton's equations in real physical time $t$**, unlocking dynamical correlation functions, collective excitations, and transport coefficients (diffusion, viscosity, thermal and electrical conductivity).

### Static Perturbation Theory & Linear Susceptibility
Suppose we perturb the potential energy $V(\mathbf{R})$ of a canonical ensemble by a weak external field $\lambda A(\mathbf{R})$:
$$H(\lambda) = V(\mathbf{R}) + \lambda A(\mathbf{R}), \qquad e^{-\beta F(\lambda)} = \int d^{3N}r\, e^{-\beta V(\mathbf{R}) - \beta \lambda A(\mathbf{R})}$$
Expanding $e^{-\beta\lambda A}$ in powers of $\beta\lambda$ and taking the logarithm yields the cumulant expansion of the free energy:
$$F(\lambda) = F(0) + \lambda \langle A \rangle_0 - \frac{1}{2}\beta \lambda^2 \left[\langle A^2 \rangle_0 - \langle A \rangle_0^2\right] + \cdots$$
Similarly, the expectation value of any observable $B(\mathbf{R})$ in the perturbed system is:
$$\langle B \rangle_\lambda = \langle B \rangle_0 - \beta \lambda \left[\langle B A \rangle_0 - \langle B \rangle_0 \langle A \rangle_0\right] + O(\lambda^2) \implies \left.\frac{\partial \langle B \rangle_\lambda}{\partial \lambda}\right|_{\lambda=0} = -\beta \langle \delta B\, \delta A \rangle_0$$
**Density Response Example:** Let the external perturbation couple to Fourier mode $\mathbf{k}$ of the density ($A = \rho_{\mathbf{k}}$) and measure the induced density wave $B = \rho_{-\mathbf{k}}$ (with $\langle \rho_{\mathbf{k}} \rangle_0 = 0$ for $\mathbf{k} \neq \mathbf{0}$ in a fluid):
$$\left.\frac{\partial \langle \rho_{-\mathbf{k}} \rangle}{\partial \lambda_{\mathbf{k}}}\right|_{\lambda=0} = -\beta \left\langle |\rho_{\mathbf{k}}|^2 \right\rangle_0 = -\beta N S(\mathbf{k})$$
Thus, the **static structure factor $S(\mathbf{k})$ is directly proportional to the static linear density susceptibility**, which is why X-ray and neutron scattering measure $S(\mathbf{k})$.

### Dynamical Correlation Functions $C_{AB}(t)$
For two observables $A$ and $B$ with fluctuations $\delta A(t) = A(t) - \langle A \rangle$, the equilibrium **time correlation function** is:
$$C_{AB}(t) = \left\langle \delta A(t_0)\, \delta B(t_0 + t) \right\rangle = \left\langle \delta A(0)\, \delta B(t) \right\rangle$$
- **Stationarity & Ergodicity:** In thermal equilibrium, ensemble averages are invariant under time translation ($t_0 \to 0$). If the system is **ergodic**, we can average over many time origins $t_0$ along a single long MD trajectory to dramatically reduce statistical noise.
- **Decorrelation at Long Times:** In an ergodic mixing system, observables become statistically independent as $t \to \infty$, so $\lim_{t\to\infty} C_{AB}(t) = \langle \delta A\rangle \langle \delta B\rangle = 0$.
- **Autocorrelation Bounds & Wiener-Khinchin Positivity ($B = A^*$):** By the Cauchy-Schwarz inequality, an autocorrelation function never exceeds its zero-time variance:
 $$\left| C_{AA}(t) \right| \le C_{AA}(0) = \langle |\delta A|^2 \rangle$$
 Moreover, by the Wiener-Khinchin theorem, its temporal Fourier transform (power spectral density) is strictly non-negative at all frequencies:
 $$C_{AA}(\omega) = \int_{-\infty}^\infty \frac{dt}{2\pi}\, e^{i\omega t} C_{AA}(t) \ge 0$$


In [3]:
ws.render_section('q01_why_md_over_mc')

**Q1 · easy** · 💡

What is the primary physical motivation for performing **Molecular Dynamics (MD)** simulations rather than equilibrium **Monte Carlo (MC)** simulations when studying transport and relaxation phenomena?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare deterministic Newtonian time evolution in MD with stochastic Markov chain trial moves in MC.

</div>
</details>

- [ ] **(A)** MD integrates the physical equations of motion in real time $t$, allowing direct evaluation of time correlation functions $C_{AB}(t)$, collective excitations $S(k, \omega)$, and transport coefficients.
- [ ] **(B)** MC cannot compute static equilibrium properties such as pressure $P$ or radial distribution function $g(r)$.
- [ ] **(C)** MD does not require an interatomic potential energy function $V(\mathbf{R})$.
- [ ] **(D)** MC simulations are restricted to 1D systems.

In [4]:
ws.render_section('q02_static_perturbation_free_energy')

**Q2 · medium** · 💡

When the Hamiltonian is perturbed by $\lambda A(\mathbf{R})$, static perturbation theory yields the Taylor expansion of the Helmholtz free energy:
$$F(\lambda) = F(0) + \lambda \langle A \rangle_0 - \frac{1}{2}\beta \lambda^2 \left[\langle A^2 \rangle_0 - \langle A \rangle_0^2\right] + \cdots$$
Since variance $\langle A^2 \rangle_0 - \langle A \rangle_0^2 = \langle (\delta A)^2 \rangle_0 \ge 0$ is always non-negative, what is the sign of the second-order free-energy shift $\Delta F^{(2)} = -\frac{1}{2}\beta \lambda^2 \langle (\delta A)^2 \rangle_0$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Note that $\beta = 1/(k_B T) > 0$, $\lambda^2 > 0$, and $\langle (\delta A)^2 \rangle_0 \ge 0$.

</div>
</details>

- [ ] **(A)** It is always negative or zero ($\Delta F^{(2)} \le 0$), meaning thermal fluctuations allow the system to relax into states that lower its free energy in response to the perturbation.
- [ ] **(B)** It is always strictly positive ($\Delta F^{(2)} > 0$).
- [ ] **(C)** Its sign alternates randomly with particle number $N$.
- [ ] **(D)** It is identically zero at all temperatures.

In [5]:
ws.render_section('q03_density_susceptibility_sk_numeric')

**Q3 · medium** · 💡

In a liquid of $N = 400$ atoms at inverse temperature $\beta = (k_B T)^{-1} = 0.50\,\varepsilon^{-1}$, the static structure factor at wavevector $k_1$ has its main peak value $S(k_1) = 2.50$. Using the static linear response relation:
$$\left.\frac{d\langle \rho_{-k_1} \rangle}{d\lambda_{k_1}}\right|_{\lambda=0} = -\beta N S(k_1)$$
calculate the magnitude $\left|\frac{d\langle \rho_{-k_1}\rangle}{d\lambda_{k_1}}\right|$ (in units of $\varepsilon^{-1}$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the product of inverse temperature, total particle number, and the peak structure factor.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q04_time_correlation_stationarity')

**Q4 · easy** · 💡

Why can an equilibrium MD simulation improve the statistical precision of a time correlation function $C_{AB}(t) = \langle \delta A(t_0)\, \delta B(t_0 + t) \rangle$ by averaging over many different starting times $t_0$ along a single trajectory?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Does an equilibrium ensemble depend on the absolute clock time $t_0$, or only on the time lag $t$?

</div>
</details>

- [ ] **(A)** Because an equilibrium system is stationary (time-translation invariant) and ergodic, so $\langle \delta A(t_0)\delta B(t_0 + t)\rangle$ depends only on the lag time $t$ and not on the time origin $t_0$.
- [ ] **(B)** Because $C_{AB}(t)$ is independent of the lag time $t$.
- [ ] **(C)** Because energy is not conserved in Molecular Dynamics.
- [ ] **(D)** Because $t_0$ must always be set to $+\infty$.

In [7]:
ws.render_section('q05_autocorrelation_properties')

**Q5 · medium** · 💡

Which of the following mathematical properties must hold for any classical equilibrium autocorrelation function $C_{AA}(t) = \langle \delta A(0)\, \delta A^*(t) \rangle$ in an ergodic mixing system? (Select all that apply.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check for the large-time limit, the upper bound at $t=0$, and the sign of the Fourier transform $C_{AA}(\omega)$.

</div>
</details>

- [ ] **(A)** Cauchy-Schwarz upper bound: $|C_{AA}(t)| \le C_{AA}(0)$ for all $t$.
- [ ] **(B)** Long-time decorrelation: $\lim_{t \to \infty} C_{AA}(t) = 0$.
- [ ] **(C)** Non-negative power spectrum: $C_{AA}(\omega) = \int_{-\infty}^\infty \frac{dt}{2\pi} e^{i\omega t} C_{AA}(t) \ge 0$ for all $\omega$.
- [ ] **(D)** Strict monotonicity: $C_{AA}(t)$ can never become negative at intermediate times $t > 0$.

In [8]:
ws.render_section('sec_fdt_and_onsager_and_scattering')

### Onsager's Regression Hypothesis & Fluctuation-Dissipation Theorem (FDT)
How does a system disturbed slightly out of equilibrium relax back toward equilibrium? Lars Onsager (1931, Nobel Prize 1968) formulated the **Onsager Regression Hypothesis**:
> *The relaxation of macroscopic non-equilibrium disturbances is governed by the exact same laws as the regression of spontaneous microscopic fluctuations in an unperturbed equilibrium system.*

Mathematically, if a weak time-dependent field $f(t) = f_\omega e^{-i\omega t}$ couples to observable $A$, the linear response of observable $B$ is $\delta\langle B(\omega)\rangle = \chi_{BA}(\omega) f_\omega$, where the **dynamical susceptibility** $\chi(\omega) = \chi'(\omega) + i\chi''(\omega)$ is given by the **Fluctuation-Dissipation Theorem (FDT)**:
$$\chi_{BA}(\omega) = \beta \int_0^\infty dt\, e^{i\omega t} \left\langle B(t) \left.\frac{dA}{dt}\right|_{t=0} \right\rangle_0$$
- **Causality & Kramers-Kronig Relations:** Because the response cannot precede the perturbation ($\chi(t < 0) = 0$), $\chi(\omega)$ is analytic in the upper half of the complex frequency plane, so its dispersive real part $\chi'(\omega)$ and dissipative imaginary part $\chi''(\omega)$ determine each other via the **Kramers-Kronig principal-value integrals**:
 $$\chi'(\omega) = \frac{1}{\pi} \mathcal{P}\!\int_{-\infty}^\infty d\omega'\, \frac{\chi''(\omega')}{\omega' - \omega}, \qquad \chi''(\omega) = -\frac{1}{\pi} \mathcal{P}\!\int_{-\infty}^\infty d\omega'\, \frac{\chi'(\omega')}{\omega' - \omega}$$
- **Energy Dissipation & Quantum FDT:** The average power absorbed by the system from a harmonic drive of amplitude $A(\omega)$ at frequency $\omega$ is proportional to the imaginary part $\chi''(\omega)$:
 $$P_{\text{diss}}(\omega) = \frac{\omega}{2}\, \chi''(\omega)\, A^2(\omega)$$
 In quantum mechanics, the dissipative susceptibility is linked to the equilibrium correlation function by:
 $$\chi''(\omega) = \frac{1}{2\hbar}\left(e^{\beta\hbar\omega} - 1\right) \int_{-\infty}^\infty dt\, e^{-i\omega t} \langle \delta B(t)\,\delta A(0)\rangle \;\;\xrightarrow{\hbar\to 0}\;\; \frac{1}{2}\beta\omega \int_{-\infty}^\infty dt\, e^{-i\omega t} \langle \delta B(t)\,\delta A(0)\rangle$$
 In inelastic scattering, the quantum factor $e^{\beta\hbar\omega}$ produces the **detailed balance asymmetry** $S(\mathbf{k}, -\omega) = e^{-\beta\hbar\omega} S(\mathbf{k}, \omega)$ between neutron/photon energy loss (Stokes, $\omega > 0$) and energy gain (anti-Stokes, $\omega < 0$).

### Intermediate Scattering Function $F(\mathbf{k}, t)$ and Dynamical Structure Factor $S(\mathbf{k}, \omega)$
To probe collective density excitations at wavevector $\mathbf{k}$, we correlate $\rho_{\mathbf{k}}(t)$ with $\rho_{-\mathbf{k}}(0)$ to form the **intermediate scattering function** $F(\mathbf{k}, t)$ (normalized by $1/N$, or $1/2$ in standard convention):
$$F(\mathbf{k}, t) = \frac{1}{N}\left\langle \rho_{\mathbf{k}}(t)\, \rho_{-\mathbf{k}}(0) \right\rangle, \qquad F(\mathbf{k}, t=0) = S(\mathbf{k})$$
Its temporal Fourier transform is the **dynamical structure factor** $S(\mathbf{k}, \omega)$, measured directly by **inelastic neutron scattering (INS)** and inelastic X-ray scattering:
$$S(\mathbf{k}, \omega) = \int_{-\infty}^\infty \frac{dt}{2\pi}\, e^{i\omega t} F(\mathbf{k}, t), \qquad \int_{-\infty}^\infty d\omega\, S(\mathbf{k}, \omega) = F(\mathbf{k}, 0) = S(\mathbf{k})$$
- **Sound Waves & Damping:** A propagating acoustic sound wave $\delta\rho(x,t) = \varepsilon \cos(qx - c_s q t)$ creates sharp **Brillouin side-peaks** in $S(k, \omega)$ at $\omega = \pm c_s k$. Viscous and thermal damping of the sound wave broadens these delta peaks into Lorentzians of width $\Gamma_s \propto k^2$, while diffusive heat/entropy relaxation produces a central **Rayleigh peak** around $\omega = 0$.


In [9]:
ws.render_section('q06_onsager_regression_hypothesis')

**Q6 · easy** · 💡

What is the central physical statement of **Onsager's regression hypothesis** (1930s, Nobel Prize 1968)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare how a small non-equilibrium perturbation relaxes back to equilibrium with how spontaneous thermal fluctuations decay inside an unperturbed equilibrium system.

</div>
</details>

- [ ] **(A)** The relaxation of a macroscopic system from a weak non-equilibrium perturbation follows the exact same time-decay law as the regression of spontaneous microscopic fluctuations in thermal equilibrium.
- [ ] **(B)** A system in thermal equilibrium never experiences any microscopic fluctuations.
- [ ] **(C)** Non-equilibrium systems always relax instantaneously in zero time.
- [ ] **(D)** Transport coefficients can only be measured at infinite temperature.

In [10]:
ws.render_section('q07_kramers_kronig_physical_origin')

**Q7 · medium** · 💡

In linear response theory, the **Kramers-Kronig relations** allow you to compute the real (dispersive) part $\chi'(\omega)$ of the dynamical susceptibility from an integral over the imaginary (dissipative) part $\chi''(\omega')$, and vice versa. What fundamental physical principle guarantees the Kramers-Kronig relations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Why is the time-domain response function $\chi(t)$ strictly zero for $t < 0$ (making $\chi(\omega)$ analytic in the upper half-plane)?

</div>
</details>

- [ ] **(A)** Causality: the response of the system cannot precede the applied perturbation in time ($\chi(t) = 0$ for $t < 0$).
- [ ] **(B)** Periodic boundary conditions in a cubic box.
- [ ] **(C)** The ideal-gas law $P\Omega = N k_B T$.
- [ ] **(D)** Absence of interatomic forces.

In [11]:
ws.render_section('q08_power_dissipation_formula')

**Q8 · easy** · 💡

When an external monochromatic field of amplitude $A(\omega)$ and frequency $\omega$ drives a system with complex susceptibility $\chi(\omega) = \chi'(\omega) + i\chi''(\omega)$, which component of $\chi(\omega)$ governs the net **power dissipation** (energy absorption per unit time, $\text{Power} = \frac{\omega}{2}\chi''(\omega) A^2(\omega)$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check for the formula for Power dissipation.

</div>
</details>

- [ ] **(A)** The imaginary (out-of-phase) part $\chi''(\omega)$.
- [ ] **(B)** The real (in-phase) part $\chi'(\omega)$.
- [ ] **(C)** The zero-frequency static susceptibility $\chi'(0)$ only.
- [ ] **(D)** Neither; linear response never dissipates energy.

In [12]:
ws.render_section('q09_quantum_classical_fdt_limit')

**Q9 · medium** · 💡

In the quantum Fluctuation-Dissipation Theorem, the prefactor relating $\chi''(\omega)$ to the correlation spectrum is $\frac{1}{2\hbar}(e^{\beta\hbar\omega} - 1)$. In the classical / low-frequency limit where $\beta\hbar\omega \ll 1$, Taylor-expanding $e^{\beta\hbar\omega} - 1 \approx \beta\hbar\omega$ reduces this prefactor to $\frac{1}{2}\beta\omega$.

What is the exact value of the dimensionless ratio $\frac{(e^{\beta\hbar\omega} - 1) / (2\hbar)}{\frac{1}{2}\beta\omega} = \frac{e^x - 1}{x}$ when $x = \beta\hbar\omega = 1.0$? (Round to two decimal places.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute the ratio $(e^x - 1)/x$ at the specified dimensionless argument.

</div>
</details>

_Numeric answer required._

In [13]:
ws.render_section('q10_detailed_balance_neutron_scattering')

**Q10 · medium** · 💡

In an inelastic neutron scattering experiment on liquid water, quantum detailed balance requires that the dynamical structure factor obey:
$$\frac{S(\mathbf{k}, +\omega)}{S(\mathbf{k}, -\omega)} = e^{+\beta\hbar\omega}$$
where $+\omega$ corresponds to neutron energy loss (creating an excitation) and $-\omega$ corresponds to neutron energy gain (absorbing a thermally populated excitation). If an excitation has energy $\hbar\omega = 2.0\,k_B T$ ($\beta\hbar\omega = 2.0$), what is the intensity ratio $S(\mathbf{k}, +\omega) / S(\mathbf{k}, -\omega)$? (Round to two decimal places.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the detailed balance relation using the ratio of excitation energy to thermal energy.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q11_f_kt_sum_rule')

**Q11 · easy** · 💡

How is the **static structure factor** $S(\mathbf{k}) = \frac{1}{N}\langle |\rho_{\mathbf{k}}|^2 \rangle$ obtained from the intermediate scattering function $F(\mathbf{k}, t) = \frac{1}{N}\langle \rho_{\mathbf{k}}(t)\rho_{-\mathbf{k}}(0)\rangle$ and the dynamical structure factor $S(\mathbf{k}, \omega) = \int_{-\infty}^\infty \frac{dt}{2\pi} e^{i\omega t} F(\mathbf{k}, t)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $F(\mathbf{k}, t)$ at $t = 0$ using inverse Fourier transformation of $S(\mathbf{k}, \omega)$.

</div>
</details>

- [ ] **(A)** $S(\mathbf{k}) = F(\mathbf{k}, t=0) = \int_{-\infty}^\infty d\omega\, S(\mathbf{k}, \omega)$
- [ ] **(B)** $S(\mathbf{k}) = \lim_{t \to \infty} F(\mathbf{k}, t) = S(\mathbf{k}, \omega=0)$
- [ ] **(C)** $S(\mathbf{k}) = \frac{d}{dt} F(\mathbf{k}, t)\big|_{t=0}$
- [ ] **(D)** $S(\mathbf{k})$ and $S(\mathbf{k}, \omega)$ are completely unrelated.

In [15]:
ws.render_section('q12_sound_wave_s_kw_peaks')

**Q12 · medium** · 💡

suppose a fluid supports an acoustic sound wave at wavevector $q$ with dispersion $\omega_s(q) = c_s q$ that undergoes viscous damping with decay rate $\Gamma_s$. How does this collective sound wave appear in the dynamical structure factor $S(q, \omega)$ as a function of frequency $\omega$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

An undamped cosine wave $e^{\pm i\omega_s t}$ gives delta peaks at $\pm \omega_s$; exponential damping $e^{-\Gamma_s t}$ broadens those peaks.

</div>
</details>

- [ ] **(A)** As a pair of broadened peaks (Brillouin doublets) centered at $\omega = \pm c_s q$, whose frequency width increases with the sound damping rate $\Gamma_s$.
- [ ] **(B)** As a single infinitely sharp delta function at $\omega = 0$ independent of $q$.
- [ ] **(C)** As a flat horizontal line $S(q, \omega) = 1$ for all $\omega$.
- [ ] **(D)** As negative spikes where $S(q, \omega) < 0$.

In [16]:
ws.render_section('q13_hard_sphere_skw_wavelength_dependence')

**Q13 · medium** · 💡

compare panel (a) at small wavevector $kd = 0.38$ (long wavelength, $\lambda \gg d$) with panel (b) at larger wavevector $kd = 2.28$ (short wavelength comparable to interparticle spacing, $\lambda \sim d$) at the same high density $V/V_0 = 1.6$. Why does the collective sound side-peak at $\omega^* \approx 0.35$ appear clearly in panel (a) ($kd = 0.38$) but disappear in panel (b) ($kd = 2.28$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Can a continuum hydrodynamic sound wave propagate when its wavelength $\lambda = 2\pi/k$ shrinks down to the individual particle diameter $d$?

</div>
</details>

- [ ] **(A)** At small $kd = 0.38$ ($\lambda \gg d$), collective hydrodynamic sound waves propagate coherently over many particle diameters; at $kd = 2.28$ ($\lambda \sim d$), scattering probes single-particle diffusive/collisional motion where sound modes are overdamped.
- [ ] **(B)** Because hard spheres do not collide at $kd = 2.28$.
- [ ] **(C)** Because the temperature is zero in panel (b).
- [ ] **(D)** Because $S(k, \omega)$ is only defined for $kd > 10$.

In [17]:
ws.render_section('sec_diffusion_einstein_and_green_kubo')

### Macroscopic Diffusion: Fick's 1st and 2nd Laws
When a tagged particle concentration gradient $\nabla \rho(\mathbf{r}, t)$ exists, **Fick's First Law** states that the diffusive particle flux $\mathbf{j}(\mathbf{r}, t)$ is proportional to the negative gradient:
$$\mathbf{j}(\mathbf{r}, t) = -D\, \nabla \rho(\mathbf{r}, t)$$
Combining Fick's First Law with local **conservation of mass** (the continuity equation $\frac{\partial \rho}{\partial t} + \nabla\cdot\mathbf{j} = 0$) yields **Fick's Second Law** (the diffusion equation):
$$\frac{\partial \rho(\mathbf{r}, t)}{\partial t} = D\, \nabla^2 \rho(\mathbf{r}, t)$$
In $d$ spatial dimensions, a point source $\rho(\mathbf{r}, 0) = \delta(\mathbf{r})$ spreads as a Gaussian packet $P(\mathbf{r}, t) = (4\pi D t)^{-d/2} \exp\!\left(-\frac{r^2}{4Dt}\right)$ whose second moment grows linearly with time:
$$\left\langle |\mathbf{r}(t) - \mathbf{r}(0)|^2 \right\rangle = \int d^d r\, r^2 P(\mathbf{r}, t) = 2 d D t$$

### Route 1: The Einstein Mean-Squared Displacement (MSD) Relation
In $d$ dimensions ($2d = 6$ in 3D, $4$ in 2D, $2$ in 1D), the self-diffusion constant $D$ is determined by the long-time slope of the **mean-squared displacement (MSD)**:
$$D = \lim_{t \to \infty} \frac{1}{2 d t} \left\langle \left|\mathbf{r}_i(t) - \mathbf{r}_i(0)\right|^2 \right\rangle \quad \xrightarrow{d=3} \quad D = \lim_{t \to \infty} \frac{1}{6 t} \left\langle \left|\mathbf{r}_i(t) - \mathbf{r}_i(0)\right|^2 \right\rangle$$
- **Short-Time Ballistic Regime ($t \ll \tau_c$):** Before colliding with neighbors, a particle moves at constant velocity $\mathbf{r}_i(t) - \mathbf{r}_i(0) \approx \mathbf{v}_i(0) t$, so $\text{MSD}(t) \approx \langle v^2 \rangle t^2 = \left(\frac{d k_B T}{m}\right) t^2$ (quadratic in $t$!).
- **CRITICAL PBC RULE — Use Unwrapped ("Unwound") Positions:** In an MD simulation with periodic boundary conditions, whenever an atom crosses a box boundary its wrapped coordinate jumps by $\pm L$. If you plug PBC-wrapped coordinates $\mathbf{r}_{\text{wrapped}} \in [0, L)^3$ into the Einstein formula, $|\mathbf{r}_{\text{wrapped}}(t) - \mathbf{r}_{\text{wrapped}}(0)|^2$ **cannot exceed $3 L^2$** and saturates to a flat plateau $\sim L^2/2$, falsely giving $D = 0$! You **must** accumulate continuous, **unwound (unwrapped)** positions $\mathbf{r}_i^{\text{unwrap}}(t) = \mathbf{r}_i(0) + \int_0^t \mathbf{v}_i(t')\, dt'$.

### Route 2: The Green-Kubo Velocity Autocorrelation (VACF) Integral
Writing the unwrapped displacement as the time integral of velocity, $\mathbf{r}_i(t) - \mathbf{r}_i(0) = \int_0^t dt'\, \mathbf{v}_i(t')$, we can prove the exact mathematical equivalence between the Einstein MSD and the **Green-Kubo** formula:
$$\left\langle |\Delta\mathbf{r}(t)|^2 \right\rangle = \int_0^t dt' \int_0^t dt''\, \left\langle \mathbf{v}_i(t')\cdot\mathbf{v}_i(t'') \right\rangle = 2 \int_0^t dt' \int_0^{t'} d\tau\, \left\langle \mathbf{v}_i(\tau)\cdot\mathbf{v}_i(0) \right\rangle$$
Differentiating with respect to $t$ and dividing by $2d$ (with $d=3$ in 3D) yields the **Green-Kubo formula for self-diffusion**:
$$D = \frac{1}{d} \int_0^\infty dt\, \left\langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0) \right\rangle \quad \xrightarrow{d=3} \quad D = \frac{1}{3} \int_0^\infty dt\, \left\langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0) \right\rangle$$
- **Dilute Gas:** Uncorrelated binary collisions cause $\langle \mathbf{v}(t)\cdot\mathbf{v}(0)\rangle = \frac{3k_B T}{m} e^{-t/\tau_c}$ to decay monotonically.
- **Dense Liquid (Cage Effect):** Near the triple point, a moving atom quickly collides with its surrounding shell of nearest neighbors and rebounds backward, producing a pronounced **negative dip** ($\langle \mathbf{v}(t)\cdot\mathbf{v}(0)\rangle < 0$) before decaying to zero.
- **Crystalline Solid:** Atoms oscillate harmonically around fixed lattice sites; positive and negative lobes of $\langle \mathbf{v}(t)\cdot\mathbf{v}(0)\rangle$ cancel so that the integral equals $D = 0$.


In [18]:
ws.render_section('concept_vacf_msd_duality')

### Interactive Explorer: VACF C_vv(t), Cage Backscattering, and Einstein MSD Duality

Explore the mathematical duality between the **Green-Kubo VACF integral** $D(t) = \frac{1}{3}\int_0^t \langle \mathbf{v}(t')\cdot\mathbf{v}(0)\rangle dt'$ and the **Einstein Mean-Squared Displacement** $\langle |\Delta\mathbf{r}(t)|^2\rangle = 6\int_0^t D(t')dt'$ across four physical regimes: **Dilute Gas**, **Dense Liquid**, **Glassy/Arrested**, and **Crystalline Solid**.

**Key Takeaway:** In a **Dense Liquid**, the negative rebound dip in $C_{vv}(t)$ causes the running diffusivity $D(t)$ to peak and then drop down to its plateau value $D_\infty$. Switching to **Einstein MSD View** shows that every regime starts out ballistically ($\propto t^2$) at very short times before transitioning to linear $6Dt$ diffusion (in gases/liquids) or a bounded vibrational plateau (in solids/glasses).

In [19]:
ws.render_section('concept_pbc_unwrapping_bug')

### Interactive Explorer: Why Einstein MSD Requires Unwound (Unwrapped) PBC Positions

One of the most frequent bugs in MD analysis scripts is computing the Einstein MSD directly from PBC-wrapped box coordinates $\mathbf{r}_i \in [0, L)^3$ or minimum-image displacements $\Delta\mathbf{r}_i \in [-L/2, L/2]^3$. Adjust the diffusivity $D$ and box size $L$ below to see what happens when particles diffuse past $L/2$.

**Key Takeaway:** Because the Green-Kubo formula integrates $\mathbf{v}_i(t)$, it implies continuous trajectories $\mathbf{r}_i^{\text{unwrap}}(t) - \mathbf{r}_i(0) = \int_0^t \mathbf{v}_i(t')dt'$. Wrapping coordinates back into the simulation box destroys long-time displacements and causes the apparent MSD to saturate at $L^2/2$ (or $L^2/4$ for minimum-image differences)!

In [20]:
ws.render_section('q14_ficks_laws_derivation')

**Q14 · easy** · 💡

which two physical equations are combined to derive Fick's Second Law (the diffusion equation $\frac{\partial \rho}{\partial t} = D \nabla^2 \rho$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the two labels next to Fick's equations: 'Linear response + Conservation of mass'.

</div>
</details>

- [ ] **(A)** Fick's First Law of linear response ($\mathbf{j} = -D\nabla\rho$) and local conservation of mass (the continuity equation $\frac{\partial \rho}{\partial t} = -\nabla\cdot\mathbf{j}$).
- [ ] **(B)** Newton's third law and the ideal gas equation of state.
- [ ] **(C)** The Debye-Waller factor and Lindemann's melting criterion.
- [ ] **(D)** Quantum detailed balance and the virial pressure theorem.

In [21]:
ws.render_section('q15_einstein_dimension_prefactor')

**Q15 · easy** · 💡

In a $d$-dimensional isotropic fluid, how do the geometric dimensionality prefactors relate the self-diffusion coefficient $D$ to the mean-squared displacement $\langle |\mathbf{r}(t) - \mathbf{r}(0)|^2 \rangle$ and the velocity autocorrelation integral $\int_0^\infty \langle \mathbf{v}(t)\cdot\mathbf{v}(0)\rangle dt$ for a **2D monolayer** ($d=2$) versus a **3D bulk fluid** ($d=3$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the number of independent spatial dimensions contributing to the total particle displacement and velocity vector dot product.

</div>
</details>

- [ ] **(A)** In 2D: $\text{MSD} = 4 D t$ and $D = \frac{1}{2}\int_0^\infty \langle \mathbf{v}(t)\cdot\mathbf{v}(0)\rangle dt$; in 3D: $\text{MSD} = 6 D t$ and $D = \frac{1}{3}\int_0^\infty \langle \mathbf{v}(t)\cdot\mathbf{v}(0)\rangle dt$.
- [ ] **(B)** In both 2D and 3D: $\text{MSD} = 6 D t$ and $D = \frac{1}{3}\int_0^\infty \langle \mathbf{v}(t)\cdot\mathbf{v}(0)\rangle dt$.
- [ ] **(C)** In 2D: $\text{MSD} = 2 D t$; in 3D: $\text{MSD} = 3 D t$.
- [ ] **(D)** In 2D: $\text{MSD} = 4 D t^2$; in 3D: $\text{MSD} = 6 D t^2$.

In [22]:
ws.render_section('q16_msd_short_time_ballistic')

**Q16 · medium** · 💡

At very short times $t \ll \tau_c$ (before a particle experiences significant collisions with its neighbors), how does the 3D mean-squared displacement $\langle |\mathbf{r}_i(t) - \mathbf{r}_i(0)|^2 \rangle$ scale with time $t$ and temperature $T$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

At $t \to 0$, $\mathbf{r}_i(t) - \mathbf{r}_i(0) \approx \mathbf{v}_i(0) t$. Use equipartition $\langle v^2 \rangle = 3 k_B T / m$.

</div>
</details>

- [ ] **(A)** Ballistically (quadratically in time): $\langle |\Delta\mathbf{r}(t)|^2 \rangle \approx \langle v^2 \rangle t^2 = \left(\frac{3 k_B T}{m}\right) t^2$.
- [ ] **(B)** Linearly in time: $\langle |\Delta\mathbf{r}(t)|^2 \rangle = 6 D t$ even at $t \to 0$.
- [ ] **(C)** Exponentially in time: $\langle |\Delta\mathbf{r}(t)|^2 \rangle \propto e^{t/\tau_c}$.
- [ ] **(D)** Inversely with time: $\langle |\Delta\mathbf{r}(t)|^2 \rangle \propto t^{-3/2}$.

In [23]:
ws.render_section('q17_einstein_msd_slope_numeric')

**Q17 · easy** · 💡

In a 3D MD simulation of molten nickel (similar to ), the unwrapped mean-squared displacement increases linearly from $\text{MSD}(2.0\,\text{ps}) = 6.2\,\text{Å}^2$ to $\text{MSD}(6.0\,\text{ps}) = 18.2\,\text{Å}^2$. Calculate the self-diffusion coefficient $D$ in units of $\text{Å}^2/\text{ps}$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the asymptotic linear growth rate of the 3D mean-squared displacement to the diffusion coefficient via the Einstein relation.

</div>
</details>

_Numeric answer required._

In [24]:
ws.render_section('q18_pbc_unwrapping_reason')

**Q18 · medium** · 💡

Why does emphasize **"Einstein relation (no PBC!) — Use 'unwound' positions to get equivalence between the 2 forms"**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What happens to $|\mathbf{r}_i(t) - \mathbf{r}_i(0)|^2$ at large $t$ if coordinates are wrapped back into $[0, L)^3$ every time a particle crosses a periodic box wall?

</div>
</details>

- [ ] **(A)** Because PBC-wrapped coordinates are confined to $[0, L)^3$, causing the wrapped MSD to artificially saturate at $\sim L^2/2$ (giving an apparent long-time slope $D = 0$), whereas $\int_0^t \mathbf{v}_i(t')dt'$ equals the continuous unwound displacement.
- [ ] **(B)** Because periodic boundary conditions violate conservation of momentum during collisions.
- [ ] **(C)** Because the Green-Kubo velocity integral only works if all velocities are set to zero at the box boundaries.
- [ ] **(D)** Because PBC-wrapped coordinates cause the MSD to grow as $t^4$.

In [25]:
ws.render_section('q19_pbc_wrapped_msd_saturation_numeric')

**Q19 · hard** · 💡

Suppose a student forgets to unwrap periodic boundary crossings in a 3D cubic box of side length $L = 10.0\,\text{Å}$ and computes the long-time limit of $\langle |\mathbf{r}_{\text{wrapped}}(t) - \mathbf{r}_{\text{wrapped}}(0)|^2 \rangle$, where $\mathbf{r}_{\text{wrapped}} \in [0, L)^3$.

As $t \to \infty$, $\mathbf{r}_{\text{wrapped}}(t)$ and $\mathbf{r}_{\text{wrapped}}(0)$ become two independent, uniformly distributed points in $[0, L)^3$. Since for a single coordinate $x \sim \text{Uniform}(0, L)$ the mean-squared difference between two independent draws is $\langle (x_1 - x_2)^2 \rangle = 2 \text{Var}(x) = 2(L^2 / 12) = L^2 / 6$, what plateau value (in $\text{Å}^2$) does the 3D wrapped MSD saturate to as $t \to \infty$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Sum the independent mean-squared coordinate separation across all three Cartesian directions.

</div>
</details>

_Numeric answer required._

In [26]:
ws.render_section('q20_vacf_t0_equipartition_numeric')

**Q20 · medium** · 💡

At lag time $t = 0$, the 3D velocity autocorrelation function $C_{vv}(0) = \langle \mathbf{v}_i(0)\cdot\mathbf{v}_i(0) \rangle = \langle v_i^2 \rangle$ is fixed purely by equilibrium statistical mechanics (independent of density or interatomic forces!). For a classical particle of mass $m = 2.0$ (in reduced units) at temperature $k_B T = 1.5$, what is the exact value of $C_{vv}(0) = \langle \mathbf{v}_i(0)\cdot\mathbf{v}_i(0) \rangle$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the mean-squared velocity of a particle to the temperature and particle mass using equipartition across the three translational degrees of freedom.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q21_vacf_exponential_green_kubo_numeric')

**Q21 · medium** · 💡

In a moderately dilute 3D gas where the velocity autocorrelation function decays exponentially as:
$$\langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0) \rangle = \left(\frac{3 k_B T}{m}\right) e^{-t / \tau_c}$$
with $\frac{k_B T}{m} = 1.20\,\text{Å}^2/\text{ps}^2$ and collision relaxation time $\tau_c = 0.50\,\text{ps}$, calculate the self-diffusion coefficient $D$ (in $\text{Å}^2/\text{ps}$) using the 3D Green-Kubo formula $D = \frac{1}{3}\int_0^\infty \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0)\rangle dt$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Integrate the exponential correlation decay over positive time and apply the Green-Kubo prefactor.

</div>
</details>

_Numeric answer required._

In [28]:
ws.render_section('q22_vacf_negative_dip_cage_effect')

**Q22 · easy** · 💡

and in the **Dense Liquid** mode of the interactive explorer, why does the velocity autocorrelation function $C_{vv}(t)$ become **negative** around $t \approx 0.3\text{–}0.5\,\text{ps}$ before decaying to zero?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about what happens when an atom in a tightly packed liquid moves a fraction of an angstrom and hits its first coordination shell.

</div>
</details>

- [ ] **(A)** Cage backscattering (rebound): at high liquid density, a moving particle collides with its tightly packed shell of nearest neighbors and rebounds in the opposite direction, reversing its velocity vector.
- [ ] **(B)** Because the kinetic temperature $T$ becomes negative at $t = 0.4\,\text{ps}$.
- [ ] **(C)** Because the mass of the Lennard-Jones atoms changes sign.
- [ ] **(D)** Because periodic boundary conditions invert the velocity vector when an atom crosses $L/2$.

In [29]:
ws.render_section('q23_vacf_fourier_spectrum_liquid')

**Q23 · medium** · 💡

Look at Fig. 2.3(b), which plots the temporal Fourier transform $\hat{c}_{vv}(\omega)$ of the velocity autocorrelation function for a dense Lennard-Jones liquid. What do (i) the non-zero intercept $\hat{c}_{vv}(\omega = 0) > 0$ and (ii) the broad peak at finite frequency $\omega \approx 25\,\text{cm}^{-1}$ physically represent?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $\int_0^\infty C_{vv}(t) e^{i\omega t} dt$ at $\omega = 0$ to connect to the Green-Kubo diffusivity $D$, and connect the finite-$\omega$ peak to the cage oscillation frequency.

</div>
</details>

- [ ] **(A)** The zero-frequency intercept $\hat{c}_{vv}(0)$ is directly proportional to the self-diffusion constant $D$, while the finite-frequency peak reflects oscillatory rattling of atoms inside their transient neighbor cages.
- [ ] **(B)** The zero-frequency intercept $\hat{c}_{vv}(0)$ is the shear viscosity $\eta$, and the peak is the ideal-gas pressure.
- [ ] **(C)** Both $\hat{c}_{vv}(0)$ and the peak height are identically zero in a liquid.
- [ ] **(D)** $\hat{c}_{vv}(0)$ measures the crystal lattice constant.

In [30]:
ws.render_section('q24_vacf_crystalline_solid_integral')

**Q24 · medium** · 💡

In a defect-free harmonic crystalline solid at thermal equilibrium, atoms vibrate around fixed lattice sites without long-range diffusive hopping, so the mean-squared displacement saturates to a bounded constant $\lim_{t\to\infty} \langle |\mathbf{r}_i(t) - \mathbf{r}_i(0)|^2 \rangle = 2\langle u^2 \rangle < \infty$.

What is the exact value of the Green-Kubo integral $\frac{1}{3}\int_0^\infty \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0) \rangle dt$ in this crystalline solid?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the long-time growth rate of the mean-squared displacement in a system without macroscopic diffusion.

</div>
</details>

_Numeric answer required._

In [31]:
ws.render_section('q25_anisotropic_diffusion_cs2')

**Q25 · medium** · 💡

(Fig. 6.5, Tildesley & Madden 1983) shows the MSD and VACF of linear triatomic carbon disulfide ($\text{CS}_2$, $\text{S=C=S}$) molecules in the liquid phase, resolved into components **parallel** ($\parallel$) and **perpendicular** ($\perp$) to the molecular axis at $t = 0$. Why does the perpendicular VACF component $c_{vv}^\perp(t)$ show a strong negative cage-rebound dip at $192\,\text{K}$ while the parallel component $c_{vv}^\parallel(t)$ decays much more smoothly and gives a steeper initial MSD slope?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the cross-sectional area of a needle-like linear molecule moving end-on (parallel to its axis) versus broadside (perpendicular to its axis).

</div>
</details>

- [ ] **(A)** Moving broadside ($\perp$) encounters a much larger steric cross-section and strong cage rebounding from neighboring molecules, whereas sliding end-on ($\parallel$) along the molecular axis experiences less steric hindrance.
- [ ] **(B)** Because a linear molecule has zero mass along its parallel axis.
- [ ] **(C)** Because equipartition is violated at $t = 0$ between parallel and perpendicular translational velocities.
- [ ] **(D)** Because $\text{CS}_2$ is a crystalline solid at $294\,\text{K}$.

In [32]:
ws.render_section('q26_liquid_metal_solute_diffusion')

**Q26 · medium** · 💡

the *ab initio* MD trajectories of molten $\text{Ni}_{400}\text{X}_{100}$ alloys ($\text{X} = \text{Al}, \text{W}, \text{Re}$) yield solute diffusion constants $D(\text{Al}) = 0.38\,\text{Å}^2/\text{ps}$, $D(\text{W}) = 0.27\,\text{Å}^2/\text{ps}$, and $D(\text{Re}) = 0.25\,\text{Å}^2/\text{ps}$, andalso show that adding $\text{W}$, $\text{Re}$, or $\text{Al}$ reduces the solvent $\text{Ni}$ diffusivity from $0.50\,\text{Å}^2/\text{ps}$ (pure $\text{Ni}$) down to $0.33\text{–}0.37\,\text{Å}^2/\text{ps}$. What physical factors explain why $\text{W}$ and $\text{Re}$ diffuse significantly more slowly than $\text{Al}$ in molten $\text{Ni}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the atomic masses and refractory transition-metal bonding of W ($Z=74, m\approx 184$) and Re ($Z=75, m\approx 186$) against light Al ($Z=13, m\approx 27$).

</div>
</details>

- [ ] **(A)** W and Re are heavy 5d refractory transition metals ($m \approx 184\text{–}186\,\text{u}$, thermal velocity $v_{\text{th}} \propto m^{-1/2}$) with strong directional d-band bonding that slows both their own hopping and the surrounding Ni solvent cage.
- [ ] **(B)** Al atoms have a larger atomic mass than W and Re.
- [ ] **(C)** A steeper MSD slope corresponds to a smaller diffusion constant $D$.
- [ ] **(D)** Solute diffusion cannot be measured using the Einstein MSD relation.

In [33]:
ws.render_section('sec_green_kubo_transport_and_hydro_tails')

### The Green-Kubo Family of Transport Coefficients
Taking the zero-frequency, long-wavelength limit ($\omega \to 0, k \to 0$) of the Fluctuation-Dissipation Theorem shows that **every linear transport coefficient $\mu$ can be written in Green-Kubo form as the time integral of a microscopic flux/current autocorrelation function**:
$$\mu \propto \int_0^\infty dt\, \left\langle J(t_0)\, J(t_0 + t) \right\rangle_0$$

| Transport Coefficient | Macroscopic Law | Microscopic Flux / Current $J(t)$ | Green-Kubo Formula (3D) |
|:--- |:--- |:--- |:--- |
| **Self-Diffusion** $D$ | $\mathbf{j} = -D\nabla\rho$ | Tagged particle velocity $\mathbf{v}_i(t)$ | $D = \frac{1}{3}\int_0^\infty dt\, \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0)\rangle$ |
| **Shear Viscosity** $\eta$ | $\sigma_{xy} = -\eta \frac{\partial u_x}{\partial y}$ | Off-diagonal stress tensor $\sigma_{xy}(t)$ | $\eta = \frac{\Omega}{k_B T} \int_0^\infty dt\, \langle \sigma_{xy}(t)\,\sigma_{xy}(0)\rangle$ |
| **Thermal Conductivity** $\lambda_T$ | $\mathbf{J}^Q = -\lambda_T \nabla T$ | Microscopic heat/energy current $\mathbf{J}^Q(t)$ | $\lambda_T = \frac{\Omega}{3 k_B T^2} \int_0^\infty dt\, \langle \mathbf{J}^Q(t)\cdot\mathbf{J}^Q(0)\rangle$ |
| **Electrical Conductivity** $\sigma_{\text{el}}$ | $\mathbf{J}^e = \sigma_{\text{el}} \mathbf{E}$ | Total electric current $\mathbf{J}(t) = \sum_i q_i \mathbf{v}_i(t)$ | $\sigma_{\text{el}} = \frac{1}{3\Omega k_B T} \int_0^\infty dt\, \langle \mathbf{J}(t)\cdot\mathbf{J}(0)\rangle$ |

### Non-Equilibrium Molecular Dynamics (NEMD) & Thermostats
Instead of integrating equilibrium fluctuations via Green-Kubo, we can also measure transport coefficients using **Non-Equilibrium MD (NEMD)** by directly imposing a finite shear rate, temperature gradient, or electric field and measuring the steady-state response flux.
- **Why NEMD Requires a Thermostat:** Driving a steady current or shear flow performs continuous mechanical/electrical work on the system, generating irreversible **Joule/viscous heating** ($P_{\text{diss}} > 0$). Without a **thermostat** to remove this heat, the total energy and temperature of the simulation would rise indefinitely and a steady state could never be reached!

### Alder–Wainwright Hydrodynamic Long-Time Tails ($t^{-d/2}$)
Before computer simulations, kinetic theory (Boltzmann/Enskog "molecular chaos") assumed that successive collisions are uncorrelated, predicting that the velocity autocorrelation function $C_{vv}(t)$ decays **exponentially** ($e^{-t/\tau_E}$) at long times.
- In 1967–1970, **Berni Alder and Thomas Wainwright** used hard-sphere MD simulations to discover a shocking result: at long times, $C_{vv}(t)$ does **not** decay exponentially, but instead exhibits a positive **algebraic power-law tail**:
 $$C_{vv}(t) = \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0) \rangle \propto t^{-d/2} \qquad (t \gg \tau_E)$$
- **Physical Mechanism (Vortex & Train Effects):** A moving particle pushes fluid ahead of it and leaves a low-pressure wake behind it. Momentum conservation in the surrounding fluid creates a pair of circulating **hydrodynamic vortices** that loop around and push the particle from behind in its original direction (**train/vortex effect**)! Because vorticity diffuses hydrodynamically over a volume $\sim (\nu t)^{d/2}$, the recirculated velocity decays as $t^{-d/2}$.
- **Catastrophic Consequence in 2D ($d = 2$):**
 - In **3D ($d = 3$)**, the tail scales as $t^{-3/2}$, whose Green-Kubo integral $\int^\infty t^{-3/2} dt < \infty$ **converges** to a finite diffusion constant $D$.
 - In **2D ($d = 2$)**, the tail scales as $t^{-2/2} = t^{-1}$, so the Green-Kubo integral $\int_0^t (t')^{-1} dt' \propto \ln(t)$ **diverges logarithmically** as $t \to \infty$ (and scales as $\ln L$ with box size)! Strictly speaking, **Fickian transport coefficients do not exist in infinite 2D fluids** because of hydrodynamic long-time tails.


In [34]:
ws.render_section('concept_alder_wainwright_explorer')

### Interactive Explorer: Alder–Wainwright Hydrodynamic Long-Time Tails (t^{-d/2})

Compare the classical Boltzmann/Enskog exponential prediction ($e^{-t/\tau_E}$) against the true **Alder–Wainwright hydrodynamic vortex tails** ($\propto t^{-d/2}$) in 2D ($t^{-1}$) and 3D ($t^{-3/2}$). Switch `panel_mode` to see how the running Green-Kubo integral $D(t)$ converges in 3D but diverges logarithmically ($\propto \ln t$) in 2D!

**Key Takeaway:** On a log-log plot, a power law $t^{-d/2}$ is a straight line with slope $-1$ (in 2D) or $-1.5$ (in 3D), completely dominating over the plunging exponential curve $e^{-t/\tau_E}$ for $t \gtrsim 10\tau_E$. Even simple hard-sphere collisions reproduce this collective hydrodynamic vortex effect!

In [35]:
ws.render_section('q27_green_kubo_flux_matching')

**Q27 · easy** · 💡

In the Green-Kubo formulation of linear transport coefficients $\mu \propto \int_0^\infty dt\, \langle A(0) A(t) \rangle$, which microscopic observable $A(t)$ is autocorrelated to obtain the **shear viscosity** $\eta$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check and for the microscopic conjugate flux paired with Viscosity.

</div>
</details>

- [ ] **(A)** The off-diagonal component of the microscopic stress (pressure) tensor $\sigma_{xy}(t)$.
- [ ] **(B)** The total electric current $\mathbf{J}^e(t) = \sum_i q_i \mathbf{v}_i(t)$.
- [ ] **(C)** The microscopic heat/energy current $\mathbf{J}^Q(t)$.
- [ ] **(D)** The static radial distribution function $g(r)$.

In [36]:
ws.render_section('q28_green_kubo_heat_and_electrical_flux')

**Q28 · easy** · 💡

Which pair of microscopic currents $(J_{\text{heat}}, J_{\text{elec}})$ are integrated in the Green-Kubo formulas for **thermal conductivity** $\lambda_T$ and **electrical conductivity** $\sigma_{\text{el}}$, respectively?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at ('Heat transport: energy current; Electrical Conductivity: electrical current').

</div>
</details>

- [ ] **(A)** Microscopic energy (heat) current $\mathbf{J}^Q(t)$ for $\lambda_T$, and total electrical current $\mathbf{J}(t) = \sum_i q_i \mathbf{v}_i(t)$ for $\sigma_{\text{el}}$.
- [ ] **(B)** Stress tensor $\sigma_{xy}(t)$ for $\lambda_T$, and particle density $\rho_{\mathbf{k}}(t)$ for $\sigma_{\text{el}}$.
- [ ] **(C)** Total potential energy $V(t)$ for both.
- [ ] **(D)** Single-particle position $\mathbf{r}_1(t)$ for both.

In [37]:
ws.render_section('q29_collective_vs_self_green_kubo')

**Q29 · medium** · 💡

Why can the self-diffusion coefficient $D$ be averaged over all $N$ individual particles ($D = \frac{1}{3N}\sum_{i=1}^N \int_0^\infty \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0)\rangle dt$, giving $N$ independent samples per snapshot), whereas the shear viscosity $\eta = \frac{\Omega}{k_B T}\int_0^\infty \langle \sigma_{xy}(t)\sigma_{xy}(0)\rangle dt$ and thermal conductivity $\lambda_T$ have much larger statistical noise for the same trajectory length?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Is $\sigma_{xy}(t)$ a single-particle property or a single collective box-wide observable?

</div>
</details>

- [ ] **(A)** Because self-diffusion is a single-particle property (allowing averaging over all $N$ atoms), whereas $\sigma_{xy}(t)$, $\mathbf{J}^Q(t)$, and $\mathbf{J}^e(t)$ are collective global observables of the entire box (providing only $1$ sample per time origin).
- [ ] **(B)** Because $\sigma_{xy}(t)$ is only defined for $N = 1$ particle.
- [ ] **(C)** Because the velocity autocorrelation function never decays to zero.
- [ ] **(D)** Because viscosity cannot be computed in 3D.

In [38]:
ws.render_section('q30_nemd_thermostat_requirement')

**Q30 · medium** · 💡

when evaluating a transport coefficient via a **non-equilibrium simulation (NEMD)**—for example, by applying a continuous external electric field $\mathbf{E}$ or continuous shear rate $\dot{\gamma}$—why is it mandatory to couple the system to a **thermostat**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What happens to the internal energy and temperature when an external force continuously does dissipative work ($P_{\text{diss}} = \sigma_{\text{el}} E^2 > 0$ or $\eta \dot{\gamma}^2 > 0$) on a closed system?

</div>
</details>

- [ ] **(A)** Because the external driving field continuously performs irreversible work (viscous or Joule heating) on the system; without a thermostat to extract this heat, the energy and temperature would drift upward indefinitely instead of reaching a steady state.
- [ ] **(B)** Because without a thermostat, all particles stop moving immediately when an electric field is applied.
- [ ] **(C)** Because a thermostat is required to break conservation of mass.
- [ ] **(D)** Because Green-Kubo integrals can only be evaluated at $T = 0\,\text{K}$.

In [39]:
ws.render_section('q31_alder_wainwright_discovery')

**Q31 · easy** · 💡

What celebrated discovery did **Alder and Wainwright** make using hard-sphere Molecular Dynamics simulations that overturned the classical Boltzmann/Enskog picture of uncorrelated binary collisions?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check and the interactive Alder-Wainwright explorer for the long-time behavior of $C_{vv}(t)$.

</div>
</details>

- [ ] **(A)** They discovered that the velocity autocorrelation function $C_{vv}(t)$ does not decay exponentially at long times, but instead exhibits a positive hydrodynamic power-law tail $C_{vv}(t) \propto t^{-d/2}$ caused by vortex recirculation.
- [ ] **(B)** They discovered that hard spheres never collide in 2D or 3D.
- [ ] **(C)** They proved that the velocity autocorrelation function is identically zero for all $t > 0$.
- [ ] **(D)** They showed that attractive Lennard-Jones tails are required to produce hydrodynamic vortices.

In [40]:
ws.render_section('q32_alder_wainwright_2d_divergence')

**Q32 · medium** · 💡

Why does state that **"The diffusion constant does not exist in 2D because of hydrodynamic effects"**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $d = 2$ into the long-time tail $C_{vv}(t) \propto t^{-d/2}$ and integrate $\int^\infty C_{vv}(t)\,dt$.

</div>
</details>

- [ ] **(A)** In 2D ($d = 2$), the hydrodynamic long-time tail decays as $C_{vv}(t) \propto t^{-2/2} = t^{-1}$, so the Green-Kubo integral $D(t) = \frac{1}{2}\int_0^t C_{vv}(t')dt' \propto \ln(t)$ diverges logarithmically as $t \to \infty$.
- [ ] **(B)** In 2D, particles cannot move at all, so $D = 0$ identically at all temperatures.
- [ ] **(C)** In 2D, $C_{vv}(t)$ grows exponentially as $e^{+t}$.
- [ ] **(D)** In 2D, velocities have no $x$ or $y$ components.

In [41]:
ws.render_section('q33_alder_wainwright_3d_exponent_numeric')

**Q33 · easy** · 💡

In a **3D** fluid ($d = 3$), the Alder–Wainwright hydrodynamic long-time tail of the velocity autocorrelation function decays as a power law: $C_{vv}(t) \propto t^{-\alpha} \qquad (t \to \infty)$. What is the numerical value of the positive decay exponent $\alpha$ in 3D?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the hydrodynamic vortex recirculation decay exponent to the spatial dimensionality of the fluid.

</div>
</details>

_Numeric answer required._

In [42]:
ws.render_section('q34_train_and_vortex_effect_physics')

**Q34 · medium** · 💡

How do the **train effect** and **vortex effect** affect the sign of the velocity autocorrelation function $C_{vv}(t)$ at long times ($t \gg \tau_E$) in a fluid of moderately low-to-intermediate density?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Does a circulating vortex wake push the tagged particle forward in the same direction as its initial velocity $\mathbf{v}_i(0)$ or backward?

</div>
</details>

- [ ] **(A)** They produce a positive tail ($C_{vv}(t) > 0$) because the momentum pushed forward by the particle recirculates via a vortex ring into the low-pressure wake behind it, continuing to push the particle in its original direction.
- [ ] **(B)** They make $C_{vv}(t)$ oscillate between $+1$ and $-1$ forever with undamped amplitude.
- [ ] **(C)** They cause the particle to freeze into a crystal.
- [ ] **(D)** They only occur if particles interact via long-range gravitational attraction.

In [43]:
ws.render_section('sec_synthesis_and_advanced_problems')

### Summary of Key Dynamical Identities
1. **Static Susceptibility & Structure Factor:**
 $$\left.\frac{\partial \langle B \rangle_\lambda}{\partial \lambda}\right|_{\lambda=0} = -\beta \langle \delta B\, \delta A \rangle_0, \qquad \left.\frac{\partial \langle \rho_{-\mathbf{k}} \rangle}{\partial \lambda_{\mathbf{k}}}\right|_{\lambda=0} = -\beta N S(\mathbf{k})$$
2. **Self-Part of Intermediate Scattering Function $F_s(\mathbf{k}, t)$ in the Diffusive Limit:**
 For a tagged diffusing particle whose displacement $\Delta\mathbf{r}(t) = \mathbf{r}_i(t) - \mathbf{r}_i(0)$ is Gaussian with $\langle |\Delta\mathbf{r}(t)|^2 \rangle = 6Dt$ (in 3D), the self-intermediate scattering function is:
 $$F_s(k, t) = \left\langle e^{i\mathbf{k}\cdot[\mathbf{r}_i(t) - \mathbf{r}_i(0)]} \right\rangle = \exp\!\left(-\frac{1}{6} k^2 \langle |\Delta\mathbf{r}(t)|^2 \rangle\right) = e^{-D k^2 t}$$
 Taking the time Fourier transform gives a **Lorentzian quasielastic peak** in $S_s(k, \omega)$ of half-width at half-maximum (HWHM) $\Gamma(k) = D k^2$:
 $$S_s(k, \omega) = \frac{1}{\pi} \frac{D k^2}{\omega^2 + (D k^2)^2}$$
3. **Einstein–Green-Kubo Equivalence & Nernst-Einstein Relation:**
 - Self-diffusion (3D): $D = \lim_{t\to\infty} \frac{1}{6t}\langle |\mathbf{r}_i^{\text{unwrap}}(t) - \mathbf{r}_i(0)|^2 \rangle = \frac{1}{3}\int_0^\infty \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0)\rangle dt$.
 - If ion velocities in an electrolyte are uncorrelated between distinct ions ($i \neq j$), the electrical conductivity Green-Kubo integral $\sigma_{\text{el}} = \frac{1}{3\Omega k_B T}\int_0^\infty \langle \mathbf{J}(t)\cdot\mathbf{J}(0)\rangle dt$ simplifies to the **Nernst-Einstein relation** $\sigma_{\text{el}} = \frac{\rho q^2 D}{k_B T}$ (equivalently, mobility $\mu = D / (k_B T)$).


In [44]:
ws.render_section('q35_self_isf_gaussian_diffusion')

**Q35 · medium** · 💡

In the hydrodynamic diffusive regime ($k \to 0, t \gg \tau_c$), a diffusing particle's displacement $\Delta\mathbf{r}(t)$ obeys Fick's Gaussian solution with $F_s(k, t) = \langle e^{i\mathbf{k}\cdot\Delta\mathbf{r}(t)} \rangle = e^{-D k^2 t}$. What is the shape and half-width at half-maximum (HWHM) $\Gamma(k)$ of the corresponding self-dynamical structure factor $S_s(k, \omega) = \int_{-\infty}^\infty \frac{dt}{2\pi} e^{i\omega t} F_s(k, t)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

The Fourier transform of a two-sided exponential $e^{-D k^2 |t|}$ is a Lorentzian $\frac{1}{\pi}\frac{D k^2}{\omega^2 + (D k^2)^2}$.

</div>
</details>

- [ ] **(A)** A Lorentzian peak centered at $\omega = 0$ whose half-width at half-maximum scales quadratically with wavevector, $\Gamma(k) = D k^2$.
- [ ] **(B)** Two sharp delta functions at $\omega = \pm D k$.
- [ ] **(C)** A Gaussian whose width is independent of $k$.
- [ ] **(D)** A step function that vanishes for all $|\omega| < D k^2$.

In [45]:
ws.render_section('q36_isf_decay_time_numeric')

**Q36 · medium** · 💡

A liquid has self-diffusion coefficient $D = 0.25\,\text{Å}^2/\text{ps}$. At wavevector $k = 0.40\,\text{Å}^{-1}$ in the diffusive regime where the self-intermediate scattering function follows $F_s(k, t) = \exp(-D k^2 t)$, what is the relaxation time $\tau_k$ (in $\text{ps}$) required for $F_s(k, t)$ to decay to $1/e$ of its initial value?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the characteristic time constant for an exponential decay governed by the diffusive wavevector decay rate.

</div>
</details>

_Numeric answer required._

In [46]:
ws.render_section('q37_green_kubo_viscosity_numeric')

**Q37 · hard** · 💡

In an equilibrium MD simulation of a fluid in a periodic box of volume $\Omega = 1000\,\text{Å}^3$ at thermal energy $k_B T = 0.80$ (in reduced energy units), you integrate the off-diagonal stress autocorrelation function from $t = 0$ to $\infty$:
$$\int_0^\infty dt\, \left\langle \sigma_{xy}(t)\, \sigma_{xy}(0) \right\rangle = 0.0024$$
(in reduced units of $\text{pressure}^2 \times \text{time}$). Using the Green-Kubo formula $\eta = \frac{\Omega}{k_B T}\int_0^\infty dt\,\langle \sigma_{xy}(t)\sigma_{xy}(0)\rangle$, calculate the shear viscosity $\eta$ in reduced units.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the Green-Kubo expression for shear viscosity using the simulation volume, thermal energy, and integrated stress correlation.

</div>
</details>

_Numeric answer required._

In [47]:
ws.render_section('q38_vacf_piecewise_integral_numeric')

**Q38 · hard** · 💡

In a 3D dense liquid, the velocity autocorrelation function $C_{vv}(t) = \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0) \rangle$ has a positive initial lobe over $0 \le t \le t_1$ with area:
$$\int_0^{t_1} C_{vv}(t)\, dt = +1.80\,\text{Å}^2/\text{ps}$$
followed by a negative cage-backscattering lobe over $t > t_1$ with area:
$$\int_{t_1}^\infty C_{vv}(t)\, dt = -0.60\,\text{Å}^2/\text{ps}$$
Using the 3D Green-Kubo relation $D = \frac{1}{3}\int_0^\infty C_{vv}(t)\, dt$, what is the asymptotic self-diffusion coefficient $D$ in $\text{Å}^2/\text{ps}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Combine the integrals over the two time intervals to find the total area under the velocity autocorrelation curve before applying the Green-Kubo prefactor.

</div>
</details>

_Numeric answer required._

In [48]:
ws.render_section('q39_nernst_einstein_electrical_conductivity')

**Q39 · hard** · 💡

The Green-Kubo formula for electrical conductivity substitutes the total electric current $\mathbf{J}(t) = \sum_{i=1}^N q_i \mathbf{v}_i(t)$ into $\sigma_{\text{el}} = \frac{1}{3\Omega k_B T}\int_0^\infty dt\, \langle \mathbf{J}(t)\cdot\mathbf{J}(0)\rangle$. Expanding the double sum over ions $i$ and $j$ gives:
$$\sigma_{\text{el}} = \frac{1}{3\Omega k_B T} \sum_{i=1}^N q_i^2 \int_0^\infty \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0)\rangle dt + \frac{1}{3\Omega k_B T} \sum_{i \neq j} q_i q_j \int_0^\infty \langle \mathbf{v}_i(t)\cdot\mathbf{v}_j(0)\rangle dt$$
If distinct ions move independently so that the cross-velocity correlations $\langle \mathbf{v}_i(t)\cdot\mathbf{v}_j(0)\rangle = 0$ for $i \neq j$, and all $N$ ions carry charge magnitude $|q_i| = q$ and have self-diffusion coefficient $D$, what does $\sigma_{\text{el}}$ simplify to?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Express the uncorrelated single-particle velocity integrals in terms of the self-diffusion coefficient and the ion number density.

</div>
</details>

- [ ] **(A)** The Nernst-Einstein relation $\sigma_{\text{el}} = \frac{\rho q^2 D}{k_B T}$ (where $\rho = N/\Omega$).
- [ ] **(B)** $\sigma_{\text{el}} = 0$ because positive and negative charges cancel in $q_i^2$.
- [ ] **(C)** $\sigma_{\text{el}} = \frac{k_B T}{\rho q^2 D}$.
- [ ] **(D)** $\sigma_{\text{el}} = N^2 q^2 D$.

In [49]:
ws.render_section('q40_finite_size_cutoff_hydro_tail')

**Q40 · hard** · 💡

In a finite periodic MD box of side length $L$ with sound speed $c_s$, up to what maximum time $t_{\max}$ can you observe the true infinite-system Alder–Wainwright hydrodynamic tail $C_{vv}(t) \propto t^{-d/2}$ before periodic boundary artifacts corrupt the correlation function?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How long does it take an acoustic compression wave traveling at speed $c_s$ to cross the periodic box of length $L$ and re-enter from the opposite side?

</div>
</details>

- [ ] **(A)** Up to the acoustic traversal time $t_{\max} \sim L / c_s$, when the hydrodynamic sound/vortex wave wraps around the periodic box and interferes with the particle.
- [ ] **(B)** Only for $t < \Delta t$ (a single MD integration step).
- [ ] **(C)** Out to $t = \infty$ regardless of how small $L$ is.
- [ ] **(D)** Only when $L = 0$.

In [50]:
ws.render_section('q41_time_correlation_symmetry_even')

**Q41 · medium** · 💡

For a classical real observable $A$ whose microscopic value is invariant (or strictly reversed) under time reversal, stationarity $\langle \delta A(0)\delta A(t)\rangle = \langle \delta A(-t)\delta A(0)\rangle = \langle \delta A(0)\delta A(-t)\rangle$ implies that the autocorrelation function $C_{AA}(t)$ is an **even function of time** ($C_{AA}(-t) = C_{AA}(t)$). What does $C_{AA}(-t) = C_{AA}(t)$ imply for its initial time derivative $\left.\frac{d C_{AA}(t)}{dt}\right|_{t=0}$ (for smooth Hamiltonian trajectories) and its Fourier transform $C_{AA}(\omega)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What is the slope at $t = 0$ of any smooth even function $f(-t) = f(t)$ (such as $\cos(\omega t)$ or $1 - a t^2$)?

</div>
</details>

- [ ] **(A)** The initial slope at $t = 0$ vanishes ($\dot{C}_{AA}(0) = 0$, so $C_{AA}(t)$ starts with a flat parabolic top $C_{AA}(0) - \frac{1}{2}\langle \dot{A}^2\rangle t^2$), and $C_{AA}(\omega)$ is purely real and even in $\omega$.
- [ ] **(B)** $C_{AA}(\omega)$ is purely imaginary and odd in $\omega$.
- [ ] **(C)** The initial value $C_{AA}(0)$ must be zero.
- [ ] **(D)** $C_{AA}(t)$ must increase linearly with $t$ near $t = 0$.

In [51]:
ws.render_section('q42_dynamical_correlations_master_summary')

**Q42 · medium** · 💡

Which of the following statements accurately summarize the core principles of dynamical correlations and transport coefficients in atomic-scale simulations? (Select all that apply.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Verify each statement against the Fluctuation-Dissipation Theorem, Einstein/Green-Kubo relations, and Alder-Wainwright tails.

</div>
</details>

- [ ] **(A)** By the Fluctuation-Dissipation Theorem and Onsager's regression hypothesis, linear transport coefficients can be computed purely from equilibrium time correlation functions without applying an external perturbation.
- [ ] **(B)** The Einstein MSD formula $D = \lim_{t\to\infty} \frac{1}{6t}\langle |\mathbf{r}_i(t) - \mathbf{r}_i(0)|^2\rangle$ and the 3D Green-Kubo VACF integral $D = \frac{1}{3}\int_0^\infty \langle \mathbf{v}_i(t)\cdot\mathbf{v}_i(0)\rangle dt$ are mathematically identical provided continuous unwound coordinates are used.
- [ ] **(C)** Inelastic neutron scattering measures the dynamical structure factor $S(\mathbf{k}, \omega)$, whose Brillouin peaks probe collective sound-wave dispersions and damping.
- [ ] **(D)** Because of Alder-Wainwright hydrodynamic vortex tails ($C_{vv}(t) \propto t^{-d/2}$), the 2D Green-Kubo integral diverges logarithmically whereas the 3D integral converges.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()